In [2]:
import pandas as pd
import numpy as np
original_df = pd.read_csv("collected_metrics.csv")
original_df['Timestamp'] = pd.to_datetime(original_df['Timestamp'], unit='ms')

print(original_df.head())
print("="*50)
print(original_df.info())

            Timestamp  Memory Usage  CPU Usage
0 2026-04-20 17:00:00     42.282991  11.566667
1 2026-04-20 17:00:15     42.291728  13.033333
2 2026-04-20 17:00:30     42.086092  12.533333
3 2026-04-20 17:00:45     42.024457  11.700000
4 2026-04-20 17:01:00     42.196733  13.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31675 entries, 0 to 31674
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Timestamp     31675 non-null  datetime64[ns]
 1   Memory Usage  31675 non-null  float64       
 2   CPU Usage     31675 non-null  float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 742.5 KB
None


In [3]:
train_df = original_df.iloc[:-300]
test_df = original_df.iloc[-300:]

print(train_df)
print("="*50)
print(test_df)

                Timestamp  Memory Usage  CPU Usage
0     2026-04-20 17:00:00     42.282991  11.566667
1     2026-04-20 17:00:15     42.291728  13.033333
2     2026-04-20 17:00:30     42.086092  12.533333
3     2026-04-20 17:00:45     42.024457  11.700000
4     2026-04-20 17:01:00     42.196733  13.000000
...                   ...           ...        ...
31370 2026-04-26 03:42:30     60.191783  15.900000
31371 2026-04-26 03:42:45     60.053899  14.966667
31372 2026-04-26 03:43:00     60.014265  13.633333
31373 2026-04-26 03:43:15     60.211084  14.866667
31374 2026-04-26 03:43:30     60.210131  15.066667

[31375 rows x 3 columns]
                Timestamp  Memory Usage  CPU Usage
31375 2026-04-26 03:43:45     60.288684  14.466667
31376 2026-04-26 03:44:00     60.298612  13.666667
31377 2026-04-26 03:44:15     60.207907  15.800000
31378 2026-04-26 03:44:30     60.356832  15.433333
31379 2026-04-26 03:44:45     60.254372  15.600000
...                   ...           ...        ...
31670

In [4]:
chronos_df = (
    train_df
    .melt(
        id_vars='Timestamp',
        value_vars=['CPU Usage', 'Memory Usage'],
        var_name='item_id',
        value_name='target'
    )
    .rename(columns={'Timestamp': 'timestamp'})
)

chronos_df['item_id'] = chronos_df['item_id'].map({
    'CPU Usage': 'CPU',
    'Memory Usage': 'Memory'
})

chronos_cpu_df = chronos_df.query("item_id == 'CPU'").reset_index(drop=True)
chronos_memory_df = chronos_df.query("item_id == 'Memory'").reset_index(drop=True)

print("CPU df:\n", chronos_cpu_df.head())
print("="*50)
print("Memory df:\n", chronos_memory_df.head())

CPU df:
             timestamp item_id     target
0 2026-04-20 17:00:00     CPU  11.566667
1 2026-04-20 17:00:15     CPU  13.033333
2 2026-04-20 17:00:30     CPU  12.533333
3 2026-04-20 17:00:45     CPU  11.700000
4 2026-04-20 17:01:00     CPU  13.000000
Memory df:
             timestamp item_id     target
0 2026-04-20 17:00:00  Memory  42.282991
1 2026-04-20 17:00:15  Memory  42.291728
2 2026-04-20 17:00:30  Memory  42.086092
3 2026-04-20 17:00:45  Memory  42.024457
4 2026-04-20 17:01:00  Memory  42.196733


In [5]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

train_data = TimeSeriesDataFrame.from_data_frame(
    chronos_df,
    id_column="item_id",
    timestamp_column="timestamp"
)

/home/nguyentan/OpenstackDRS/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
predictor = TimeSeriesPredictor(
    prediction_length=300,
    target="target",
    eval_metric="MASE",
).fit(
    train_data=train_data,
    hyperparameters={
        "Chronos2": [
            {"ag_args": {"name_suffix": "ZeroShot"}},
            {"fine_tune": True, "ag_args": {"name_suffix": "FineTuned"}},
        ]
    },
    time_limit=3600,
    enable_ensemble=False,
)

Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/home/nguyentan/OpenstackDRS/app/predictor/AutogluonModels/ag-20260509_174426'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          3
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 2.00/2.00 GB
Total GPU Memory:   Free: 2.00 GB, Allocated: 0.00 GB, Total: 2.00 GB
GPU Count:          1
Memory Avail:       0.65 GB / 3.83 GB (17.1%)
Disk Space Avail:   903.44 GB / 1006.85 GB (89.7%)

Fitting with arguments:
{'enable_ensemble': False,
 'eval_metric': MASE,
 'hyperparameters': {'Chronos2': [{'ag_args': {'name_suffix': 'ZeroShot'}},
                                  {'ag_args': {'name_suffix': 'FineTuned'},
                                   'fine_tune': True}]},
 'known_covariates